In [1]:
!pip3 install funasr modelscope

In [2]:
from funasr import AutoModel

model = AutoModel(
    model="emotion2vec/emotion2vec_plus_large",
    model_revision="v2.0.4",
    hub="hf"  # use HuggingFace instead of ModelScope
)

funasr version: 1.3.1.
Check update of funasr, and it would cost few times. You may disable it by set `disable_update=True` in AutoModel
You are using the latest version of funasr-1.3.1


Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 76260.07it/s]


Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.0.0.weight, /Users/abey/.cache/huggingface/hub/models--emotion2vec--emotion2vec_plus_large/snapshots/6c303ba987b86b93193de93e34bb2b077a6bedc4/model.pt
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.0.0.bias, /Users/abey/.cache/huggingface/hub/models--emotion2vec--emotion2vec_plus_large/snapshots/6c303ba987b86b93193de93e34bb2b077a6bedc4/model.pt
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.1.0.weight, /Users/abey/.cache/huggingface/hub/models--emotion2vec--emotion2vec_plus_large/snapshots/6c303ba987b86b93193de93e34bb2b077a6bedc4/model.pt
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.1.0.bias, /Users/abey/.cache/huggingface/hub/models--emotion2vec--emotion2vec_plus_large/snapshots/6c303ba987b86b93193de93e34bb2b077a6bedc4/model.pt
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.2.0.weight, /Users/abey/.cache/huggingface/hub/models--emotion2vec--em

In [3]:
# ============================================================
# CELL 1 — Imports
# ============================================================
import os
import pandas as pd
from funasr import AutoModel

print("✅ Imports done")

✅ Imports done


In [4]:
# ============================================================
# CELL 2 — Load emotion2vec model
# ============================================================
ser_model = AutoModel(
    model="emotion2vec/emotion2vec_plus_large",
    model_revision="v2.0.4",
    hub="hf",
    disable_update=True
)

print("✅ emotion2vec loaded")

funasr version: 1.3.1.


Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 69775.85it/s]


Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.0.0.weight, /Users/abey/.cache/huggingface/hub/models--emotion2vec--emotion2vec_plus_large/snapshots/6c303ba987b86b93193de93e34bb2b077a6bedc4/model.pt
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.0.0.bias, /Users/abey/.cache/huggingface/hub/models--emotion2vec--emotion2vec_plus_large/snapshots/6c303ba987b86b93193de93e34bb2b077a6bedc4/model.pt
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.1.0.weight, /Users/abey/.cache/huggingface/hub/models--emotion2vec--emotion2vec_plus_large/snapshots/6c303ba987b86b93193de93e34bb2b077a6bedc4/model.pt
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.1.0.bias, /Users/abey/.cache/huggingface/hub/models--emotion2vec--emotion2vec_plus_large/snapshots/6c303ba987b86b93193de93e34bb2b077a6bedc4/model.pt
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.2.0.weight, /Users/abey/.cache/huggingface/hub/models--emotion2vec--em

In [5]:
# ============================================================
# CELL 3 — Paths and constants
# ============================================================
BASE_DIR      = "/Users/abey/Documents/SER"
REFERENCE_DIR = os.path.join(BASE_DIR, "reference")
MODELS_DIR    = os.path.join(BASE_DIR, "models")

# minimum confidence to trust a prediction
# below this the label is too uncertain to use for comparison
CONFIDENCE_THRESHOLD = 0.5

print(f"BASE_DIR      : {BASE_DIR}")
print(f"REFERENCE_DIR : {REFERENCE_DIR}")
print(f"MODELS_DIR    : {MODELS_DIR}")
print(f"Min confidence: {CONFIDENCE_THRESHOLD}")
print("✅ Paths and constants set")

BASE_DIR      : /Users/abey/Documents/SER
REFERENCE_DIR : /Users/abey/Documents/SER/reference
MODELS_DIR    : /Users/abey/Documents/SER/models
Min confidence: 0.5
✅ Paths and constants set


In [6]:

# ============================================================
# CELL 4 — get_emotion function
# ============================================================
def get_emotion(audio_path):
    try:
        res = ser_model.generate(
            audio_path,
            granularity="utterance",
            extract_embedding=False,
            disable_update=True
        )
        labels = res[0]['labels']
        scores = res[0]['scores']

        # find highest scoring label
        max_idx    = scores.index(max(scores))
        raw_label  = labels[max_idx]
        confidence = round(scores[max_idx], 4)

        # strip Chinese prefix — "生气/angry" → "angry"
        clean_label = raw_label.split('/')[-1]

        return clean_label, confidence

    except Exception as e:
        print(f"  ⚠️  emotion2vec error on {audio_path}: {e}")
        return None, None

print("✅ get_emotion defined")


✅ get_emotion defined


In [7]:
# ============================================================
# CELL 5 — Startup validation
# ============================================================

# ── check models folder ──
if not os.path.exists(MODELS_DIR):
    raise FileNotFoundError(f"Models folder not found: {MODELS_DIR}")

# ── check reference folder ──
if not os.path.exists(REFERENCE_DIR):
    raise FileNotFoundError(
        f"Reference folder not found: {REFERENCE_DIR}\n"
        f"SER requires Hindi reference audio to compare against."
    )

ref_files = sorted([f for f in os.listdir(REFERENCE_DIR) if f.endswith(".wav")])
print(f"✅ Reference folder found: {len(ref_files)} files")

# ── discover model folders ──
model_folders = sorted([
    d for d in os.listdir(MODELS_DIR)
    if os.path.isdir(os.path.join(MODELS_DIR, d))
])
if not model_folders:
    raise ValueError(f"No model folders found in {MODELS_DIR}")
print(f"✅ Models found: {model_folders}")

# ── discover wav files per model ──
model_samples = {}
for model in model_folders:
    model_path = os.path.join(MODELS_DIR, model)
    wav_files  = sorted([
        f for f in os.listdir(model_path)
        if f.endswith(".wav")
    ])
    model_samples[model] = wav_files
    print(f"   {model}: {len(wav_files)} samples")

# ── validate all models have identical filenames ──
reference_filenames = set(model_samples[model_folders[0]])
for model in model_folders[1:]:
    current_filenames = set(model_samples[model])
    if current_filenames != reference_filenames:
        missing = reference_filenames - current_filenames
        extra   = current_filenames - reference_filenames
        raise ValueError(
            f"Model '{model}' has mismatched filenames.\n"
            f"  Missing : {missing}\n"
            f"  Extra   : {extra}"
        )
print("✅ All models have identical filenames")

sample_names = model_samples[model_folders[0]]
total        = len(model_folders) * len(sample_names)
print(f"\nReady: {len(model_folders)} models × {len(sample_names)} samples = {total} evaluations")

✅ Reference folder found: 2 files
✅ Models found: ['m1', 'm2']
   m1: 2 samples
   m2: 2 samples
✅ All models have identical filenames

Ready: 2 models × 2 samples = 4 evaluations


In [8]:
# ============================================================
# CELL 6 — Main evaluation loop
# ============================================================
results = []

for model in model_folders:
    print(f"\n{'='*50}")
    print(f"Model: {model}")
    print(f"{'='*50}")

    for wav_file in model_samples[model]:
        sample_name = os.path.splitext(wav_file)[0]
        tts_path    = os.path.join(MODELS_DIR, model, wav_file)
        ref_path    = os.path.join(REFERENCE_DIR, wav_file)

        print(f"\n  Sample : {sample_name}")

        # ── check reference exists ──
        if not os.path.exists(ref_path):
            print(f"  ⚠️  No reference file found — skipping")
            results.append({
                "Model"       : model,
                "Sample"      : sample_name,
                "Ref Label"   : None,
                "Ref Conf"    : None,
                "TTS Label"   : None,
                "TTS Conf"    : None,
                "Match"       : None,
                "Pass"        : "⚠️ SKIP",
                "Flag"        : "NO_REF",
                "_is_degraded": False,  # no ref = absolute only, not degraded
            })
            continue

        # ── score reference ──
        ref_label, ref_conf = get_emotion(ref_path)
        print(f"  Ref    : {ref_label} ({ref_conf})")

        # ── score TTS ──
        tts_label, tts_conf = get_emotion(tts_path)
        print(f"  TTS    : {tts_label} ({tts_conf})")

        # ── determine flag and degraded status ──
        if ref_label is None or tts_label is None:
            flag        = "ERROR"
            match       = None
            passed      = "⚠️ ERROR"
            is_degraded = False
        elif ref_conf < CONFIDENCE_THRESHOLD:
            # reference label untrustworthy — score it but put in degraded
            flag        = "LOW_CONF_REF"
            match       = ref_label == tts_label
            passed      = "✅ PASS" if match else "❌ FAIL"
            is_degraded = True
        else:
            flag        = "—"
            match       = ref_label == tts_label
            passed      = "✅ PASS" if match else "❌ FAIL"
            is_degraded = False

        print(f"  Result : {passed} | Match: {match} | Flag: {flag}")

        results.append({
            "Model"       : model,
            "Sample"      : sample_name,
            "Ref Label"   : ref_label,
            "Ref Conf"    : ref_conf,
            "TTS Label"   : tts_label,
            "TTS Conf"    : tts_conf,
            "Match"       : match,
            "Pass"        : passed,
            "Flag"        : flag,
            "_is_degraded": is_degraded,
        })

print("\n\nAll evaluations complete.")




Model: m1

  Sample : clean_baseline 2


rtf_avg: 0.106: 100%|██████████| 1/1 [00:00<00:00,  1.39it/s]                                                                                      


  Ref    : disgusted (0.9999)


rtf_avg: 0.038: 100%|██████████| 1/1 [00:00<00:00,  3.85it/s]                                                                                      


  TTS    : disgusted (0.9999)
  Result : ✅ PASS | Match: True | Flag: —

  Sample : clean_baseline


...

  TTS    : disgusted (0.9999)
  Result : ✅ PASS | Match: True | Flag: —

  Sample : clean_baseline


rtf_avg: 0.044: 100%|██████████| 1/1 [00:00<00:00,  3.33it/s]                                                                                      


  Ref    : disgusted (0.9999)


rtf_avg: 0.043: 100%|██████████| 1/1 [00:00<00:00,  3.43it/s]                                                                                      

  TTS    : disgusted (0.9999)
  Result : ✅ PASS | Match: True | Flag: —


All evaluations complete.


In [9]:

# ============================================================
# CELL 7 — Results and model comparison
# ============================================================
df = pd.DataFrame(results)

# ── Table 1 — full per segment results ──
print("\n========== FULL PER-SEGMENT RESULTS ==========")
print(df[[
    "Model", "Sample", "Ref Label", "Ref Conf",
    "TTS Label", "TTS Conf", "Pass", "Flag"
]].to_string(index=False))

# ── Table 2 — per model summary ──
print("\n========== MODEL COMPARISON SUMMARY ==========")
summary_rows = []

for model in model_folders:
    model_df    = df[df["Model"] == model]
    clean_df    = model_df[~model_df["_is_degraded"] & (model_df["Flag"] != "NO_REF") & (model_df["Flag"] != "ERROR")]
    degraded_df = model_df[model_df["_is_degraded"]]
    total       = len(model_df)

    # clean pass rate — primary ranking number
    clean_total = len(clean_df)
    clean_pass  = (clean_df["Pass"] == "✅ PASS").sum()

    # degraded pass rate — LOW_CONF_REF segments
    deg_total   = len(degraded_df)
    deg_pass    = (degraded_df["Pass"] == "✅ PASS").sum()

    # most common mismatch — what emotion did TTS produce when it failed
    fail_df = clean_df[clean_df["Pass"] == "❌ FAIL"]
    most_common_mismatch = fail_df["TTS Label"].value_counts().index[0] if len(fail_df) > 0 else "—"

    summary_rows.append({
        "Model"             : model,
        "Total Segments"    : total,
        "Clean Segments"    : clean_total,
        "Clean Pass Rate"   : f"{clean_pass}/{clean_total}"  if clean_total > 0 else "—",
        "Degraded Segments" : deg_total,
        "Degraded Pass Rate": f"{deg_pass}/{deg_total}"      if deg_total > 0 else "—",
        "Common Mismatch"   : most_common_mismatch,
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

# ── Table 3 — model ranking ──
print("\n========== MODEL RANKING ==========")
print("Primary   → Clean Pass Rate (ref confidence >= 0.5 segments only)")
print("Tiebreak1 → Degraded Pass Rate\n")

def parse_rate(rate_str):
    if rate_str == "—":
        return -1
    return int(rate_str.split("/")[0])

summary_df["_clean_pass_num"]    = summary_df["Clean Pass Rate"].apply(parse_rate)
summary_df["_degraded_pass_num"] = summary_df["Degraded Pass Rate"].apply(parse_rate)

ranking = summary_df.sort_values(
    by=["_clean_pass_num", "_degraded_pass_num"],
    ascending=[False, False]
)[[
    "Model", "Clean Pass Rate", "Degraded Pass Rate", "Common Mismatch"
]]

print(ranking.to_string(index=False))

print("\n========== WHAT TO LOOK FOR ==========")
print("Clean Pass Rate    → primary ranking — ref confidence >= 0.5 segments only")
print("Degraded Pass Rate → segments where reference emotion was ambiguous")
print("Common Mismatch    → what emotion TTS produces when it fails")
print("                     neutral is most common — TTS is emotionally flat")
print("LOW_CONF_REF       → reference confidence below 0.5 — counted in degraded")
print("NO_REF             → no reference file found — excluded from both rates")



========== FULL PER-SEGMENT RESULTS ==========
Model           Sample Ref Label  Ref Conf TTS Label  TTS Conf   Pass Flag
   m1 clean_baseline 2 disgusted    0.9999 disgusted    0.9999 ✅ PASS    —
   m1   clean_baseline disgusted    0.9999 disgusted    0.9999 ✅ PASS    —
   m2 clean_baseline 2 disgusted    0.9999 disgusted    0.9999 ✅ PASS    —
   m2   clean_baseline disgusted    0.9999 disgusted    0.9999 ✅ PASS    —

========== MODEL COMPARISON SUMMARY ==========
Model  Total Segments  Clean Segments Clean Pass Rate  Degraded Segments Degraded Pass Rate Common Mismatch
   m1               2               2             2/2                  0                  —               —
   m2               2               2             2/2                  0                  —               —

========== MODEL RANKING ==========
Primary   → Clean Pass Rate (ref confidence >= 0.5 segments only)
Tiebreak1 → Degraded Pass Rate

Model Clean Pass Rate Degraded Pass Rate Common Mismatch
   m1        

In [10]:
# final cell in each gate notebook
df.to_csv(os.path.join(BASE_DIR, "results.csv"), index=False)
print("✅ Results saved to results.csv")

✅ Results saved to results.csv
